In [1]:
import os
import pandas as pd
import shutil

# ========================= CONFIG =========================
BASE_PATH = "/kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Preprocessed_dataset"

WORKING_CSV_DIR = "/kaggle/working/shuffled_csvs"
os.makedirs(WORKING_CSV_DIR, exist_ok=True)

config = {
    "base_path": BASE_PATH,
    "train_dir": os.path.join(BASE_PATH, "train"),
    "val_dir": os.path.join(BASE_PATH, "val"),
    "test_dir": os.path.join(BASE_PATH, "test"),
    "shuffled_csvs_dir": WORKING_CSV_DIR,
    
    "train_csv": os.path.join(WORKING_CSV_DIR, "train_shuffled.csv"),
    "val_csv": os.path.join(WORKING_CSV_DIR, "val_shuffled.csv"),
    "test_csv": os.path.join(WORKING_CSV_DIR, "test_shuffled.csv"),
}

print("✅ Config ready (using writable folder)")

# ======================= STRONG FIX FUNCTION =======================
def fix_csv_robust(input_path, output_path):
    old_prefix = "/content/drive/MyDrive/TAL_GRN_Preprocessed_dataset/"
    new_prefix = BASE_PATH + "/"
    
    if not os.path.exists(input_path):
        print(f"❌ Input file not found: {input_path}")
        return False
    
    shutil.copy2(input_path, output_path)
    print(f"📋 Copied to working dir: {output_path}")
    
    # Read with more flexibility
    try:
        df = pd.read_csv(output_path, delimiter=",", quotechar='"', on_bad_lines='warn')
    except:
        df = pd.read_csv(output_path, delimiter=",", on_bad_lines='skip')
    
    print(f"   Shape: {df.shape} | Columns: {list(df.columns)}")
    
    fixed_count = 0
    for col in df.columns:
        if df[col].dtype == 'object':
            # Strong replace - handle partial matches too
            mask = df[col].str.contains(old_prefix, na=False)
            if mask.any():
                df.loc[mask, col] = df.loc[mask, col].str.replace(old_prefix, new_prefix, regex=False)
                fixed_count += mask.sum()
                print(f"   ✅ Fixed {mask.sum()} paths in column '{col}'")
    
    # Extra safety: replace anywhere in the whole file as text if needed
    if fixed_count == 0:
        print("   ⚠️ No paths found via pandas, trying raw text replace...")
        with open(output_path, 'r', encoding='utf-8') as f:
            content = f.read()
        if old_prefix in content:
            content = content.replace(old_prefix, new_prefix)
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(content)
            print(f"   ✅ Raw text replacement done on whole file")
            fixed_count = 1
    
    if fixed_count > 0:
        df.to_csv(output_path, index=False)
        print(f"   ✅ Successfully saved fixed CSV")
    else:
        print(f"   ℹ️ No old paths detected")
    
    return True

# ======================= APPLY FIX TO ALL CSVs =======================
original_base = os.path.join(BASE_PATH, "shuffled_csvs")

for split in ['train', 'val', 'test']:
    orig_file = os.path.join(original_base, f"{split}_shuffled.csv")
    new_file = config[f"{split}_csv"]
    print(f"\n🔧 Fixing {split}_shuffled.csv ...")
    fix_csv_robust(orig_file, new_file)

# ======================= VERIFICATION =======================
def verify_fix(csv_path):
    if not os.path.exists(csv_path):
        print(f"❌ CSV not found: {csv_path}")
        return
    df = pd.read_csv(csv_path)
    print(f"\n📊 {os.path.basename(csv_path)}")
    print(f"   Rows: {len(df)} | Columns: {list(df.columns)}")
    
    # Show sample path
    for col in df.columns:
        if df[col].dtype == 'object':
            sample = df[col].iloc[0] if len(df) > 0 else None
            if sample:
                print(f"   Sample from '{col}': {str(sample)[:120]}...")
                if "/content/drive" in str(sample):
                    print("   ❌ Still contains old path!")
                elif BASE_PATH in str(sample):
                    print("   ✅ Path looks correct now")
                break

verify_fix(config["test_csv"])
verify_fix(config["train_csv"])

print("\n✅ Done! Now use the `config` dictionary in your training code.")

✅ Config ready (using writable folder)

🔧 Fixing train_shuffled.csv ...
📋 Copied to working dir: /kaggle/working/shuffled_csvs/train_shuffled.csv
   Shape: (23234, 5) | Columns: ['image_path', 'label', 'split', 'filename', 'label_id']
   ✅ Fixed 23234 paths in column 'image_path'
   ✅ Successfully saved fixed CSV

🔧 Fixing val_shuffled.csv ...
📋 Copied to working dir: /kaggle/working/shuffled_csvs/val_shuffled.csv
   Shape: (4085, 5) | Columns: ['image_path', 'label', 'split', 'filename', 'label_id']
   ✅ Fixed 4085 paths in column 'image_path'
   ✅ Successfully saved fixed CSV

🔧 Fixing test_shuffled.csv ...
📋 Copied to working dir: /kaggle/working/shuffled_csvs/test_shuffled.csv
   Shape: (6873, 5) | Columns: ['image_path', 'label', 'split', 'filename', 'label_id']
   ✅ Fixed 6873 paths in column 'image_path'
   ✅ Successfully saved fixed CSV

📊 test_shuffled.csv
   Rows: 6873 | Columns: ['image_path', 'label', 'split', 'filename', 'label_id']
   Sample from 'image_path': /kaggle/inp

In [2]:
# ─────────────────────────────────────────
# CELL: Preprocess — Rebuild CSVs from Filesystem Scan
# Fixes Unicode folder name mismatches (ö encoding)
# ─────────────────────────────────────────
import os
import unicodedata
import pandas as pd
from pathlib import Path

BASE_PATH   = "/kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Preprocessed_dataset"
CSV_DIR     = "/kaggle/working/shuffled_csvs"
OUT_DIR     = "/kaggle/working/fixed_csvs"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Step 1: Walk filesystem, map filename → real absolute path ──
print("Scanning filesystem...")
fname_to_realpath = {}   # {normalized_filename: real_path}
duplicate_fnames  = []

for root, dirs, files in os.walk(BASE_PATH):
    for f in files:
        if not f.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tiff')):
            continue
        real_path  = os.path.join(root, f)
        norm_fname = unicodedata.normalize('NFC', f)   # normalize key

        if norm_fname in fname_to_realpath:
            duplicate_fnames.append(norm_fname)
        else:
            fname_to_realpath[norm_fname] = real_path

print(f"  Files found on disk : {len(fname_to_realpath):,}")
print(f"  Duplicate filenames : {len(duplicate_fnames):,}")

# ── Step 2: Fix each CSV ────────────────────────────────────────
splits = {
    'train': os.path.join(CSV_DIR, 'train_shuffled.csv'),
    'val'  : os.path.join(CSV_DIR, 'val_shuffled.csv'),
    'test' : os.path.join(CSV_DIR, 'test_shuffled.csv'),
}

IMG_COL   = 'image_path'
LABEL_COL = 'label'

for split, csv_path in splits.items():
    df = pd.read_csv(csv_path)
    print(f"\n── {split.upper()} ({len(df):,} rows) ──")

    fixed, still_missing, skipped = [], [], []

    for _, row in df.iterrows():
        old_path = str(row[IMG_COL])
        fname    = unicodedata.normalize('NFC', os.path.basename(old_path))

        if os.path.exists(old_path):
            # Already correct
            fixed.append({IMG_COL: old_path, LABEL_COL: row[LABEL_COL]})

        elif fname in fname_to_realpath:
            # Found via filesystem scan — use real path
            fixed.append({IMG_COL: fname_to_realpath[fname],
                          LABEL_COL: row[LABEL_COL]})

        else:
            # Truly missing
            still_missing.append(old_path)

    out_path = os.path.join(OUT_DIR, f'{split}_fixed.csv')
    fixed_df = pd.DataFrame(fixed)
    fixed_df.to_csv(out_path, index=False)

    print(f"  Resolved    : {len(fixed):,}")
    print(f"  Still missing: {len(still_missing):,}")
    print(f"  Saved → {out_path}")

    if still_missing:
        miss_out = os.path.join(OUT_DIR, f'still_missing_{split}.txt')
        with open(miss_out, 'w') as mf:
            mf.write('\n'.join(still_missing))
        print(f"  Missing list → {miss_out}")
        print(f"  First 3 missing: {still_missing[:3]}")

# ── Step 3: Verify sample paths ────────────────────────────────
print(f"\n{'='*60}")
print("  VERIFICATION")
print(f"{'='*60}")
for split in ['train','val','test']:
    p = os.path.join(OUT_DIR, f'{split}_fixed.csv')
    df = pd.read_csv(p)
    sample = df[IMG_COL].iloc[0]
    exists = os.path.exists(sample)
    print(f"  {split:5s} | rows={len(df):,} | sample_exists={exists} | {sample[:80]}...")

print(f"\n  ✓ Update cfg paths to point at /kaggle/working/fixed_csvs/")
print(f"    cfg.TRAIN_CSV = '{OUT_DIR}/train_fixed.csv'")
print(f"    cfg.VAL_CSV   = '{OUT_DIR}/val_fixed.csv'")
print(f"    cfg.TEST_CSV  = '{OUT_DIR}/test_fixed.csv'")

Scanning filesystem...
  Files found on disk : 34,198
  Duplicate filenames : 0

── TRAIN (23,234 rows) ──
  Resolved    : 23,234
  Still missing: 0
  Saved → /kaggle/working/fixed_csvs/train_fixed.csv

── VAL (4,085 rows) ──
  Resolved    : 4,085
  Still missing: 0
  Saved → /kaggle/working/fixed_csvs/val_fixed.csv

── TEST (6,873 rows) ──
  Resolved    : 6,873
  Still missing: 0
  Saved → /kaggle/working/fixed_csvs/test_fixed.csv

  VERIFICATION
  train | rows=23,234 | sample_exists=True | /kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Prep...
  val   | rows=4,085 | sample_exists=True | /kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Prep...
  test  | rows=6,873 | sample_exists=True | /kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Prep...

  ✓ Update cfg paths to point at /kaggle/working/fixed_csvs/
    cfg.TRAIN_CSV = '/kaggle/working/fixed_csvs/train_fixed.csv'
    cfg.VAL_CSV   = '/kaggle/working

In [3]:
# ══════════════════════════════════════════════════════════════
#  CELL 0: COMMON — imports, config, data, backbone, graph utils,
#          GNN layers, metrics, train/eval/test, seed runner.
#  Self-contained. Run this once, then run any method cell below.
# ══════════════════════════════════════════════════════════════
import os, time, random, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, matthews_corrcoef, classification_report)
from sklearn.preprocessing import label_binarize
from tqdm import tqdm

# ── Config (SAME paths/metrics as TAL-GRN) ────────────────────
class Config:
    TRAIN_CSV = "/kaggle/working/shuffled_csvs/train_shuffled.csv"
    VAL_CSV   = "/kaggle/working/shuffled_csvs/val_shuffled.csv"
    TEST_CSV  = "/kaggle/working/shuffled_csvs/test_shuffled.csv"
    IMAGE_ROOT = ""
    IMG_COL, LABEL_COL = "image_path", "label"

    BACKBONE       = "resnet50"
    FEATURE_DIM    = 2048
    GNN_HIDDEN_DIM = 512
    GNN_LAYERS     = 3
    NUM_CLASSES    = None

    BATCH_SIZE  = 64
    EPOCHS      = 50
    LR          = 1e-3
    WEIGHT_DECAY= 1e-4
    IMG_SIZE    = 224
    SEED        = 42
    NUM_WORKERS = 2
    CHECKPOINT_DIR = "/kaggle/working/baseline_checkpoints"
    DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

# ── Dataset (identical to TAL-GRN) ────────────────────────────
class ImageDataset(Dataset):
    def __init__(self, csv_path, image_root, img_col, label_col,
                 transform=None, label_map=None, img_size=224):
        self.df = pd.read_csv(csv_path)
        self.image_root, self.img_col, self.label_col = image_root, img_col, label_col
        self.transform, self.img_size = transform, img_size
        if label_map is None:
            uniq = sorted(self.df[label_col].unique())
            self.label_map = {v:i for i,v in enumerate(uniq)}
        else:
            self.label_map = label_map
        self.labels = [self.label_map[l] for l in self.df[label_col]]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        p = os.path.join(self.image_root, self.df.iloc[idx][self.img_col])
        try: img = Image.open(p).convert("RGB")
        except Exception: img = Image.new("RGB",(self.img_size,self.img_size))
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def get_transforms(split="train"):
    mean,std = [0.485,0.456,0.406],[0.229,0.224,0.225]
    if split=="train":
        return transforms.Compose([
            transforms.Resize((cfg.IMG_SIZE+32, cfg.IMG_SIZE+32)),
            transforms.RandomCrop(cfg.IMG_SIZE),
            transforms.RandomHorizontalFlip(),
            transforms.ColorJitter(0.3,0.3,0.3,0.1),
            transforms.ToTensor(), transforms.Normalize(mean,std)])
    return transforms.Compose([
        transforms.Resize((cfg.IMG_SIZE,cfg.IMG_SIZE)),
        transforms.ToTensor(), transforms.Normalize(mean,std)])

def build_loaders():
    tr = ImageDataset(cfg.TRAIN_CSV, cfg.IMAGE_ROOT, cfg.IMG_COL, cfg.LABEL_COL,
                      get_transforms("train"), img_size=cfg.IMG_SIZE)
    lm = tr.label_map; cfg.NUM_CLASSES = len(lm)
    va = ImageDataset(cfg.VAL_CSV, cfg.IMAGE_ROOT, cfg.IMG_COL, cfg.LABEL_COL,
                      get_transforms("val"), lm, cfg.IMG_SIZE)
    te = ImageDataset(cfg.TEST_CSV, cfg.IMAGE_ROOT, cfg.IMG_COL, cfg.LABEL_COL,
                      get_transforms("val"), lm, cfg.IMG_SIZE)
    tdl = DataLoader(tr, cfg.BATCH_SIZE, shuffle=True,  num_workers=cfg.NUM_WORKERS, pin_memory=True)
    vdl = DataLoader(va, cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
    sdl = DataLoader(te, cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
    print(f"  Train:{len(tr)} Val:{len(va)} Test:{len(te)} | Classes:{cfg.NUM_CLASSES}")
    return tdl, vdl, sdl, lm

# ── Frozen backbone (identical) ───────────────────────────────
class FrozenBackbone(nn.Module):
    def __init__(self, arch="resnet50"):
        super().__init__()
        base = getattr(models, arch)(pretrained=True)
        self.features = nn.Sequential(*list(base.children())[:-1])
        for p in self.features.parameters(): p.requires_grad = False
    def forward(self, x): return self.features(x).flatten(1)

# ── Fixed-k kNN graph builders (shared by ALL baselines) ──────
BASELINE_K = 5

def build_knn_graph(h, k=BASELINE_K, add_self=True, sym=True, norm=True):
    """Sym-normalized cosine kNN adj. Sparse (N,N)."""
    N = h.size(0); dev = h.device
    k = min(k, max(N-1,1))
    hn = F.normalize(h, p=2, dim=-1)
    S = hn @ hn.T
    Sns = S.clone(); Sns.fill_diagonal_(-2.0)
    _, idx = torch.topk(Sns, k, dim=1)
    rows = torch.arange(N, device=dev).unsqueeze(1).expand(N,k).reshape(-1)
    cols = idx.reshape(-1)
    w = torch.ones(rows.numel(), device=dev)
    if sym:
        rows,cols = torch.cat([rows,cols]), torch.cat([cols,rows])
        w = torch.cat([w,w])
    if add_self:
        s = torch.arange(N, device=dev)
        rows,cols = torch.cat([rows,s]), torch.cat([cols,s])
        w = torch.cat([w, torch.ones(N, device=dev)])
    A = torch.sparse_coo_tensor(torch.stack([rows,cols]), w, (N,N)).coalesce()
    if norm:
        Ad = A.to_dense(); deg = Ad.sum(1)
        di = torch.zeros_like(deg); pos = deg>0; di[pos]=deg[pos].pow(-0.5)
        Ad = di.unsqueeze(1)*Ad*di.unsqueeze(0)
        return torch.nan_to_num(Ad,0.,0.,0.).to_sparse()
    return A

def build_rw_graph(h, k=BASELINE_K, add_self=True):
    """Row-normalized (mean-agg) adj for GraphSAGE."""
    A = build_knn_graph(h, k, add_self=add_self, sym=True, norm=False).to_dense()
    A = A / A.sum(1, keepdim=True).clamp(min=1.0)
    return torch.nan_to_num(A,0.,0.,0.).to_sparse()

# ── GNN layers ────────────────────────────────────────────────
class GCNLayer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin=nn.Linear(i,o); self.act=nn.GELU(); self.norm=nn.LayerNorm(o)
    def forward(self, h, adj):
        h = self.lin(h)
        with torch.cuda.amp.autocast(enabled=False):
            h = torch.sparse.mm(adj.float(), h.float())
        return self.norm(self.act(h))

class SAGELayer(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin=nn.Linear(i*2,o); self.act=nn.GELU(); self.norm=nn.LayerNorm(o)
    def forward(self, h, adj):
        with torch.cuda.amp.autocast(enabled=False):
            agg = torch.sparse.mm(adj.float(), h.float())
        agg = agg.to(h.dtype)
        h = self.lin(torch.cat([h, agg], -1))
        return F.normalize(self.norm(self.act(h)), p=2, dim=-1)

# ── Generic backbone+GNN model (GCN/SAGE share this) ──────────
class BaselineGNN(nn.Module):
    def __init__(self, num_classes, layer_cls, graph_fn):
        super().__init__()
        d,H = cfg.FEATURE_DIM, cfg.GNN_HIDDEN_DIM
        self.backbone = FrozenBackbone(cfg.BACKBONE)
        self.graph_fn = graph_fn
        dims = [d] + [H]*cfg.GNN_LAYERS
        self.layers = nn.ModuleList([layer_cls(dims[i],dims[i+1]) for i in range(cfg.GNN_LAYERS)])
        self.res = nn.ModuleList([nn.Linear(dims[i],dims[i+1],bias=False) if dims[i]!=dims[i+1]
                                  else nn.Identity() for i in range(cfg.GNN_LAYERS)])
        self.classifier = nn.Linear(H, num_classes)
    def forward(self, x, return_features=False):
        h = self.backbone(x)
        A = self.graph_fn(h.detach())
        for layer,res in zip(self.layers, self.res):
            h = layer(h, A) + res(h)
        logits = self.classifier(h)
        return (logits, h) if return_features else logits

# ── Metrics (identical suite) ─────────────────────────────────
def compute_metrics(y, p, probs, ncls, light=False):
    m = {}
    m["accuracy"] = accuracy_score(y,p)
    m["f1_macro"] = f1_score(y,p,average="macro",zero_division=0)
    if light:
        for k in ["precision_macro","recall_macro","precision_weighted","recall_weighted",
                  "f1_weighted","roc_auc","roc_auc_ovr","roc_auc_ovo","mcc"]:
            m[k]=float("nan")
        m["f1_per_class"]=np.array([]); return m
    m["precision_macro"]    = precision_score(y,p,average="macro",zero_division=0)
    m["recall_macro"]       = recall_score(y,p,average="macro",zero_division=0)
    m["precision_weighted"] = precision_score(y,p,average="weighted",zero_division=0)
    m["recall_weighted"]    = recall_score(y,p,average="weighted",zero_division=0)
    m["f1_weighted"]        = f1_score(y,p,average="weighted",zero_division=0)
    m["f1_per_class"]       = f1_score(y,p,average=None,zero_division=0)
    try:
        if ncls==2:
            m["roc_auc"]=roc_auc_score(y,probs[:,1]); m["roc_auc_ovr"]=float("nan"); m["roc_auc_ovo"]=float("nan")
        else:
            yb = label_binarize(y, classes=list(range(ncls)))
            m["roc_auc_ovr"]=roc_auc_score(yb,probs,average="macro",multi_class="ovr")
            m["roc_auc_ovo"]=roc_auc_score(yb,probs,average="macro",multi_class="ovo")
            m["roc_auc"]=m["roc_auc_ovr"]
    except Exception:
        m["roc_auc"]=m["roc_auc_ovr"]=m["roc_auc_ovo"]=float("nan")
    m["mcc"]=matthews_corrcoef(y,p)
    return m

@torch.no_grad()
def evaluate(model, loader, criterion, ncls, split="Val", light=False):
    model.eval(); tot=0.0; L=[]; P=[]; PR=[]
    bar = tqdm(loader, desc=f"  {split:>5}", leave=False, bar_format="{l_bar}{bar:30}{r_bar}")
    for imgs,labels in bar:
        imgs,labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
        logits = model(imgs)
        tot += criterion(logits,labels).item()*imgs.size(0)
        pr = F.softmax(logits,-1).cpu().numpy()
        PR.append(pr); P.extend(pr.argmax(1)); L.extend(labels.cpu().numpy())
    L,P,PR = np.array(L),np.array(P),np.vstack(PR)
    m = compute_metrics(L,P,PR,ncls,light=light); m["loss"]=tot/len(loader.dataset)
    return m, L, P, PR

def print_metrics(m, split="Val", epoch=None):
    tag = f" [Ep {epoch}] " if epoch else " "
    print(f"\n{'─'*52}\n  {split.upper()}{tag}METRICS\n{'─'*52}")
    print(f"  {'Loss':<28}{m['loss']:.4f}")
    print(f"  {'Accuracy':<28}{m['accuracy']*100:.2f}%")
    print(f"  {'Precision (Macro)':<28}{m['precision_macro']:.4f}")
    print(f"  {'Recall (Macro)':<28}{m['recall_macro']:.4f}")
    print(f"  {'F1 (Macro)':<28}{m['f1_macro']:.4f}")
    print(f"  {'F1 (Weighted)':<28}{m['f1_weighted']:.4f}")
    print(f"  {'ROC-AUC (OvR)':<28}{m['roc_auc_ovr']:.4f}")
    print(f"  {'ROC-AUC (OvO)':<28}{m['roc_auc_ovo']:.4f}")
    print(f"  {'MCC':<28}{m['mcc']:.4f}\n{'─'*52}")

# ── Train loop (identical AMP loop; pluggable loss + step fn) ──
def default_train_step(model, imgs, labels, criterion):
    """Returns (loss, logits_for_acc, labels_for_acc). Override per-method."""
    logits = model(imgs)
    return criterion(logits, labels), logits, labels

def train_one_epoch(model, loader, optimizer, criterion, epoch, scaler, step_fn):
    model.train(); tot=0.0; corr=0; n=0
    bar = tqdm(loader, desc=f"  Epoch {epoch:>3}", bar_format="{l_bar}{bar:35}{r_bar}")
    for step,(imgs,labels) in enumerate(bar):
        imgs,labels = imgs.to(cfg.DEVICE), labels.to(cfg.DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            loss, logits_acc, labels_acc = step_fn(model, imgs, labels, criterion)
        if not torch.isfinite(loss):
            optimizer.zero_grad(); continue
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        tot += loss.item()*imgs.size(0)
        corr += (logits_acc.detach().argmax(1)==labels_acc).sum().item(); n += labels_acc.size(0)
        bar.set_postfix({"loss":f"{loss.item():.4f}","acc":f"{100.*corr/max(n,1):.1f}%"})
    return tot/max(n,1), corr/max(n,1)

def train(model, train_dl, val_dl, criterion=None, step_fn=default_train_step):
    if criterion is None: criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(filter(lambda p:p.requires_grad, model.parameters()),
                                  lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())
    best_f1 = 0.0
    hist = {"train_loss":[],"train_acc":[],"val_loss":[],"val_acc":[],"val_f1":[]}
    print(f"\n{'='*52}\n  Training {cfg.EPOCHS} epochs\n{'='*52}")
    for epoch in range(1, cfg.EPOCHS+1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion, epoch, scaler, step_fn)
        heavy = (epoch % 5 == 0)
        vm,_,_,_ = evaluate(model, val_dl, nn.CrossEntropyLoss(), cfg.NUM_CLASSES, "Val", light=not heavy)
        scheduler.step()
        for k,v in [("train_loss",tr_loss),("train_acc",tr_acc),("val_loss",vm["loss"]),
                    ("val_acc",vm["accuracy"]),("val_f1",vm["f1_macro"])]:
            hist[k].append(v)
        print(f"  Ep {epoch:>3}/{cfg.EPOCHS}  TrLoss={tr_loss:.4f} TrAcc={tr_acc*100:.1f}%  "
              f"ValLoss={vm['loss']:.4f} ValF1={vm['f1_macro']:.4f} ValAcc={vm['accuracy']*100:.1f}%  "
              f"LR={scheduler.get_last_lr()[0]:.2e} [{time.time()-t0:.0f}s]")
        if vm["f1_macro"] > best_f1:
            best_f1 = vm["f1_macro"]
            torch.save({"epoch":epoch,"model":model.state_dict(),"best_f1":best_f1},
                       os.path.join(cfg.CHECKPOINT_DIR,"best_model.pt"))
            print(f"  ★ best F1={best_f1:.4f}")
        if heavy: print_metrics(vm,"Val",epoch)
    return hist

def final_test(model, test_dl, label_map):
    bc = os.path.join(cfg.CHECKPOINT_DIR,"best_model.pt")
    if os.path.exists(bc):
        ck = torch.load(bc, map_location=cfg.DEVICE)
        model.load_state_dict(ck["model"])
        print(f"  Loaded best (ep {ck['epoch']}, F1={ck['best_f1']:.4f})")
    tm,L,P,_ = evaluate(model, test_dl, nn.CrossEntropyLoss(), cfg.NUM_CLASSES, "Test")
    print_metrics(tm,"TEST")
    inv = {v:k for k,v in label_map.items()}
    names = [inv[i] for i in range(cfg.NUM_CLASSES)]
    print("\n  CLASSIFICATION REPORT")
    for ln in classification_report(L,P,target_names=names,zero_division=0).split("\n"):
        print("  "+ln)
    return tm

# ── Seed runner + summary ─────────────────────────────────────
SEEDS = [42]
METRIC_KEYS = ["accuracy","precision_macro","recall_macro","f1_macro",
               "f1_weighted","roc_auc_ovr","roc_auc_ovo","mcc","loss"]
BASELINE_RESULTS = {}

def run_baseline(name, build_model_fn, make_criterion=None, step_fn=default_train_step):
    """build_model_fn(ncls)->model. make_criterion(train_dl,label_map)->loss (optional)."""
    runs = []
    for ri,seed in enumerate(SEEDS):
        print(f"\n{'#'*56}\n#  {name}  |  RUN {ri+1}/{len(SEEDS)}  |  SEED={seed}\n{'#'*56}")
        cfg.SEED=seed; cfg.NUM_CLASSES=None
        cfg.CHECKPOINT_DIR = f"/kaggle/working/{name}_checkpoints/seed_{seed}"
        os.makedirs(cfg.CHECKPOINT_DIR, exist_ok=True)
        set_seed(seed)
        train_dl,val_dl,test_dl,lm = build_loaders()
        model = build_model_fn(cfg.NUM_CLASSES).to(cfg.DEVICE)
        print(f"  Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
        crit = make_criterion(train_dl, lm) if make_criterion else None
        train(model, train_dl, val_dl, criterion=crit, step_fn=step_fn)
        tm = final_test(model, test_dl, lm)
        runs.append({"seed":seed, **tm})
        print(f"  ✓ {name} seed={seed} Acc={tm['accuracy']*100:.2f}% F1={tm['f1_macro']:.4f} "
              f"AUC={tm['roc_auc_ovr']:.4f} MCC={tm['mcc']:.4f}")
        del model
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    summarize(name, runs); BASELINE_RESULTS[name]=runs
    return runs

def summarize(name, runs):
    print(f"\n{'='*64}\n  {name}  ·  SUMMARY OVER {len(runs)} SEEDS\n{'='*64}")
    print(f"  {'Metric':<20}{'Mean':<11}{'Median':<11}{'Std':<11}{'Min':<11}{'Max':<11}")
    print("  "+"─"*70)
    for k in METRIC_KEYS:
        v = np.array([r.get(k,float('nan')) for r in runs]); v=v[~np.isnan(v)]
        if len(v)==0: print(f"  {k.upper():<20}N/A"); continue
        sc = 100 if k=="accuracy" else 1; sx = "%" if k=="accuracy" else ""
        print(f"  {k.upper():<20}{np.mean(v)*sc:.2f}{sx:<7}{np.median(v)*sc:.2f}{sx:<7}"
              f"{np.std(v)*sc:.2f}{sx:<7}{np.min(v)*sc:.2f}{sx:<7}{np.max(v)*sc:.2f}{sx}")
    print(f"{'='*64}\n")

print("COMMON cell loaded. Run any method cell below.")

COMMON cell loaded. Run any method cell below.


In [4]:
# ══════════════════════════════════════════════════════════════
#  CELL 1: GCN   (Kipf & Welling, ICLR 2017)
#  sym-normalized kNN adj + GCN layers
# ══════════════════════════════════════════════════════════════
def build_gcn(ncls):
    return BaselineGNN(ncls, GCNLayer,
                       lambda h: build_knn_graph(h, k=BASELINE_K, sym=True, norm=True))

run_baseline("GCN", build_gcn)


########################################################
#  GCN  |  RUN 1/1  |  SEED=42
########################################################
  Train:23234 Val:4085 Test:6873 | Classes:40
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]


  Trainable params: 2,646,568

  Training 50 epochs


  Epoch   1: 100%|███████████████████████████████████| 364/364 [03:22<00:00,  1.79it/s, loss=3.8975, acc=42.1%]


  Ep   1/50  TrLoss=2.1271 TrAcc=42.1%  ValLoss=1.4405 ValF1=0.5446 ValAcc=56.2%  LR=9.99e-04 [225s]
  ★ best F1=0.5446


...

  Ep  49/50  TrLoss=0.3783 TrAcc=86.2%  ValLoss=0.4961 ValF1=0.8055 ValAcc=81.9%  LR=9.87e-07 [122s]


  Epoch  50: 100%|███████████████████████████████████| 364/364 [01:45<00:00,  3.44it/s, loss=3.1528, acc=86.3%]


  Ep  50/50  TrLoss=0.3763 TrAcc=86.3%  ValLoss=0.4599 ValF1=0.8185 ValAcc=83.3%  LR=0.00e+00 [119s]
  ★ best F1=0.8185

────────────────────────────────────────────────────
  VAL [Ep 50] METRICS
────────────────────────────────────────────────────
  Loss                        0.4599
  Accuracy                    83.33%
  Precision (Macro)           0.8268
  Recall (Macro)              0.8228
  F1 (Macro)                  0.8185
  F1 (Weighted)               0.8279
  ROC-AUC (OvR)               0.9953
  ROC-AUC (OvO)               0.9953
  MCC                         0.8289
────────────────────────────────────────────────────
  Loaded best (ep 50, F1=0.8185)



────────────────────────────────────────────────────
  TEST METRICS
────────────────────────────────────────────────────
  Loss                        0.4549
  Accuracy                    83.20%
  Precision (Macro)           0.8287
  Recall (Macro)              0.8234
  F1 (Macro)                  0.8207
  F1 (Weighted)               0.8271
  ROC-AUC (OvR)               0.9957
  ROC-AUC (OvO)               0.9957
  MCC                         0.8275
────────────────────────────────────────────────────

  CLASSIFICATION REPORT
                                                                                            precision    recall  f1-score   support
  
                                                                 Acute Cerebellitis in HIV       0.64      0.70      0.67       196
                                                      Acute Unilateral Cerebellitis in HIV       0.40      0.12      0.19        99
                                                              Adenom

[{'seed': 42,
  'accuracy': 0.8319511130510694,
  'f1_macro': 0.8207158595548382,
  'precision_macro': 0.8287171090932581,
  'recall_macro': 0.8233511224024344,
  'precision_weighted': 0.8310161360685974,
  'recall_weighted': 0.8319511130510694,
  'f1_weighted': 0.8270673175819512,
  'f1_per_class': array([0.66828087, 0.18604651, 0.96      , 0.82582583, 0.3963964 ,
         0.94468085, 0.9512605 , 0.85470085, 0.95852535, 0.904     ,
         0.5       , 0.96259352, 0.78552972, 0.95189873, 0.9512894 ,
         0.80263158, 0.75129534, 0.94594595, 0.92703863, 0.51941748,
         0.56213018, 0.83804627, 0.9382716 , 0.83835616, 0.91079812,
         0.85338346, 0.96638655, 0.89546351, 0.96170213, 0.86774942,
         0.76296296, 0.79411765, 0.95016611, 1.        , 0.92111959,
         0.98245614, 0.64864865, 0.75085324, 0.72727273, 0.91139241]),
  'roc_auc_ovr': np.float64(0.9956564546964047),
  'roc_auc_ovo': np.float64(0.9956564546964047),
  'roc_auc': np.float64(0.9956564546964047),
  'm

In [5]:
# ══════════════════════════════════════════════════════════════
#  CELL 2: GraphSAGE   (Hamilton et al., NeurIPS 2017)
#  row-normalized (mean-agg) kNN adj + SAGE layers
# ══════════════════════════════════════════════════════════════
def build_sage(ncls):
    return BaselineGNN(ncls, SAGELayer,
                       lambda h: build_rw_graph(h, k=BASELINE_K))

run_baseline("GraphSAGE", build_sage)


########################################################
#  GraphSAGE  |  RUN 1/1  |  SEED=42
########################################################
  Train:23234 Val:4085 Test:6873 | Classes:40
  Trainable params: 4,219,432

  Training 50 epochs


  Epoch   1: 100%|███████████████████████████████████| 364/364 [01:55<00:00,  3.14it/s, loss=1.9375, acc=45.3%]


  Ep   1/50  TrLoss=1.9140 TrAcc=45.3%  ValLoss=1.4917 ValF1=0.5398 ValAcc=55.0%  LR=9.99e-04 [130s]
  ★ best F1=0.5398


  Epoch   2: 100%|███████████████████████████████████| 364/364 [01:45<00:00,  3.47it/s, loss=0.4395, acc=57.8%]


  Ep   2/50  TrLoss=1.3652 TrAcc=57.8%  ValLoss=1.1792 ValF1=0.6126 ValAcc=62.9%  LR=9.96e-04 [118s]
  ★ best F1=0.6126


...

  Ep  49/50  TrLoss=0.3937 TrAcc=85.9%  ValLoss=0.4728 ValF1=0.8125 ValAcc=82.7%  LR=9.87e-07 [121s]
  ★ best F1=0.8125


  Epoch  50: 100%|███████████████████████████████████| 364/364 [01:38<00:00,  3.70it/s, loss=2.0262, acc=85.8%]


  Ep  50/50  TrLoss=0.3937 TrAcc=85.8%  ValLoss=0.5043 ValF1=0.8063 ValAcc=82.1%  LR=0.00e+00 [112s]

────────────────────────────────────────────────────
  VAL [Ep 50] METRICS
────────────────────────────────────────────────────
  Loss                        0.5043
  Accuracy                    82.11%
  Precision (Macro)           0.8201
  Recall (Macro)              0.8111
  F1 (Macro)                  0.8063
  F1 (Weighted)               0.8150
  ROC-AUC (OvR)               0.9947
  ROC-AUC (OvO)               0.9947
  MCC                         0.8164
────────────────────────────────────────────────────
  Loaded best (ep 49, F1=0.8125)



────────────────────────────────────────────────────
  TEST METRICS
────────────────────────────────────────────────────
  Loss                        0.4746
  Accuracy                    82.79%
  Precision (Macro)           0.8238
  Recall (Macro)              0.8191
  F1 (Macro)                  0.8155
  F1 (Weighted)               0.8217
  ROC-AUC (OvR)               0.9952
  ROC-AUC (OvO)               0.9952
  MCC                         0.8233
────────────────────────────────────────────────────

  CLASSIFICATION REPORT
                                                                                            precision    recall  f1-score   support
  
                                                                 Acute Cerebellitis in HIV       0.62      0.77      0.69       196
                                                      Acute Unilateral Cerebellitis in HIV       0.35      0.08      0.13        99
                                                              Adenom

[{'seed': 42,
  'accuracy': 0.8278772006401862,
  'f1_macro': 0.8154912739877244,
  'precision_macro': 0.823801504054589,
  'recall_macro': 0.8190620353918776,
  'precision_weighted': 0.8258953420265626,
  'recall_weighted': 0.8278772006401862,
  'f1_weighted': 0.8216880129403589,
  'f1_per_class': array([0.68649886, 0.13114754, 0.97178683, 0.82882883, 0.43835616,
         0.92640693, 0.95174709, 0.86307054, 0.94392523, 0.94560669,
         0.41333333, 0.95833333, 0.78987342, 0.95854922, 0.96952909,
         0.77522936, 0.72264631, 0.9559322 , 0.93220339, 0.49767442,
         0.60923077, 0.85714286, 0.93902439, 0.83116883, 0.88151659,
         0.83941606, 0.9707113 , 0.89587426, 0.95762712, 0.84233261,
         0.70921986, 0.76470588, 0.95454545, 1.        , 0.94545455,
         0.96028881, 0.65236052, 0.73026316, 0.72      , 0.89808917]),
  'roc_auc_ovr': np.float64(0.9952143144995965),
  'roc_auc_ovo': np.float64(0.9952143144995965),
  'roc_auc': np.float64(0.9952143144995965),
  'mc

In [6]:
# ══════════════════════════════════════════════════════════════
#  CELL 3: GraphSMOTE   (Zhao et al., WSDM 2021)
#  GCN encoder + embedding-space SMOTE on minority classes each batch.
#  (Latent-interpolation variant; edge-generator decoder omitted —
#   standard simplification for inductive mini-batch image graphs.)
# ══════════════════════════════════════════════════════════════
class GraphSMOTE(nn.Module):
    def __init__(self, ncls, smote_ratio=1.0):
        super().__init__()
        d,H = cfg.FEATURE_DIM, cfg.GNN_HIDDEN_DIM
        self.backbone = FrozenBackbone(cfg.BACKBONE)
        dims = [d]+[H]*cfg.GNN_LAYERS
        self.layers = nn.ModuleList([GCNLayer(dims[i],dims[i+1]) for i in range(cfg.GNN_LAYERS)])
        self.res = nn.ModuleList([nn.Linear(dims[i],dims[i+1],bias=False) if dims[i]!=dims[i+1]
                                  else nn.Identity() for i in range(cfg.GNN_LAYERS)])
        self.classifier = nn.Linear(H, ncls)
        self.smote_ratio = smote_ratio
    def encode(self, x):
        h = self.backbone(x)
        A = build_knn_graph(h.detach(), k=BASELINE_K, sym=True, norm=True)
        for layer,res in zip(self.layers, self.res):
            h = layer(h, A) + res(h)
        return h
    def smote(self, h, y):
        nh,ny = [h],[y]
        cls,cnt = torch.unique(y, return_counts=True)
        if len(cnt)<2: return h,y
        maj = cnt.max().item()
        for c,n in zip(cls.tolist(), cnt.tolist()):
            need = int((maj-n)*self.smote_ratio)
            if need<=0 or n<2: continue
            idx = (y==c).nonzero(as_tuple=True)[0]
            a = idx[torch.randint(0,n,(need,),device=h.device)]
            b = idx[torch.randint(0,n,(need,),device=h.device)]
            lam = torch.rand(need,1,device=h.device)
            nh.append(h[a]+lam*(h[b]-h[a]))
            ny.append(torch.full((need,),c,device=h.device,dtype=y.dtype))
        return torch.cat(nh,0), torch.cat(ny,0)
    def forward(self, x, y=None, return_features=False):
        h = self.encode(x)
        if self.training and y is not None:
            h,y = self.smote(h,y)
            return self.classifier(h), y
        logits = self.classifier(h)
        return (logits,h) if return_features else logits

def smote_step(model, imgs, labels, criterion):
    logits, y_aug = model(imgs, y=labels)
    loss = criterion(logits, y_aug)
    b = labels.size(0)                       # acc on real nodes only
    return loss, logits[:b], labels

run_baseline("GraphSMOTE", lambda ncls: GraphSMOTE(ncls), step_fn=smote_step)


########################################################
#  GraphSMOTE  |  RUN 1/1  |  SEED=42
########################################################
  Train:23234 Val:4085 Test:6873 | Classes:40
  Trainable params: 2,646,568

  Training 50 epochs


  Epoch   1: 100%|███████████████████████████████████| 364/364 [01:51<00:00,  3.25it/s, loss=3.4375, acc=41.0%]


  Ep   1/50  TrLoss=2.1206 TrAcc=41.0%  ValLoss=1.4479 ValF1=0.5319 ValAcc=54.6%  LR=9.99e-04 [126s]
  ★ best F1=0.5319


  Epoch   2: 100%|███████████████████████████████████| 364/364 [02:18<00:00,  2.62it/s, loss=5.2266, acc=56.8%]


  Ep   2/50  TrLoss=1.3468 TrAcc=56.8%  ValLoss=1.3271 ValF1=0.5757 ValAcc=60.8%  LR=9.96e-04 [152s]
  ★ best F1=0.5757


...

  Ep  49/50  TrLoss=0.3534 TrAcc=85.4%  ValLoss=0.5193 ValF1=0.8011 ValAcc=81.5%  LR=9.87e-07 [105s]


  Epoch  50: 100%|███████████████████████████████████| 364/364 [01:40<00:00,  3.62it/s, loss=3.1587, acc=85.6%]


  Ep  50/50  TrLoss=0.3504 TrAcc=85.6%  ValLoss=0.4775 ValF1=0.8172 ValAcc=83.3%  LR=0.00e+00 [114s]
  ★ best F1=0.8172

────────────────────────────────────────────────────
  VAL [Ep 50] METRICS
────────────────────────────────────────────────────
  Loss                        0.4775
  Accuracy                    83.33%
  Precision (Macro)           0.8284
  Recall (Macro)              0.8220
  F1 (Macro)                  0.8172
  F1 (Weighted)               0.8275
  ROC-AUC (OvR)               0.9951
  ROC-AUC (OvO)               0.9951
  MCC                         0.8289
────────────────────────────────────────────────────
  Loaded best (ep 50, F1=0.8172)



────────────────────────────────────────────────────
  TEST METRICS
────────────────────────────────────────────────────
  Loss                        0.4722
  Accuracy                    83.08%
  Precision (Macro)           0.8271
  Recall (Macro)              0.8234
  F1 (Macro)                  0.8201
  F1 (Weighted)               0.8261
  ROC-AUC (OvR)               0.9954
  ROC-AUC (OvO)               0.9954
  MCC                         0.8263
────────────────────────────────────────────────────

  CLASSIFICATION REPORT
                                                                                            precision    recall  f1-score   support
  
                                                                 Acute Cerebellitis in HIV       0.63      0.71      0.67       196
                                                      Acute Unilateral Cerebellitis in HIV       0.43      0.12      0.19        99
                                                              Adenom

[{'seed': 42,
  'accuracy': 0.8307871380765314,
  'f1_macro': 0.8201444411537752,
  'precision_macro': 0.8271324379153864,
  'recall_macro': 0.8234064998813617,
  'precision_weighted': 0.8299306203812666,
  'recall_weighted': 0.8307871380765314,
  'f1_weighted': 0.8261165856899036,
  'f1_per_class': array([0.66825776, 0.18897638, 0.96296296, 0.82228916, 0.4526749 ,
         0.94017094, 0.94754653, 0.84388186, 0.94930876, 0.90322581,
         0.51311953, 0.97      , 0.78328982, 0.94923858, 0.95480226,
         0.79120879, 0.75703325, 0.94557823, 0.91983122, 0.51741294,
         0.55520505, 0.84974093, 0.93251534, 0.83243243, 0.90047393,
         0.84469697, 0.97478992, 0.90039841, 0.96610169, 0.87238979,
         0.75912409, 0.77155172, 0.95737705, 1.        , 0.91878173,
         0.97526502, 0.63348416, 0.73509934, 0.73359073, 0.91194969]),
  'roc_auc_ovr': np.float64(0.9954390016508892),
  'roc_auc_ovo': np.float64(0.9954390016508892),
  'roc_auc': np.float64(0.9954390016508892),
  'm

In [7]:
# ══════════════════════════════════════════════════════════════
#  CELL 4: ReNode   (Chen et al., NeurIPS 2021)
#  "Topology-Imbalance Learning for Semi-Supervised Node Classification"
#  GCN encoder + per-node ReNode reweighting (topology conflict via
#  label-propagation Personalized PageRank on the batch graph) combined
#  with class-frequency reweighting. Cosine weight schedule [w_min,w_max].
# ══════════════════════════════════════════════════════════════
class ReNodeGCN(nn.Module):
    """GCN encoder exposing batch adjacency for PPR-based ReNode weights."""
    def __init__(self, ncls):
        super().__init__()
        d,H = cfg.FEATURE_DIM, cfg.GNN_HIDDEN_DIM
        self.backbone = FrozenBackbone(cfg.BACKBONE)
        dims=[d]+[H]*cfg.GNN_LAYERS
        self.layers = nn.ModuleList([GCNLayer(dims[i],dims[i+1]) for i in range(cfg.GNN_LAYERS)])
        self.res = nn.ModuleList([nn.Linear(dims[i],dims[i+1],bias=False) if dims[i]!=dims[i+1]
                                  else nn.Identity() for i in range(cfg.GNN_LAYERS)])
        self.classifier = nn.Linear(H, ncls)
        self.last_adj = None
    def forward(self, x, return_features=False):
        h = self.backbone(x)
        A = build_knn_graph(h.detach(), k=BASELINE_K, sym=True, norm=True)
        self.last_adj = A                       # cache for ReNode weights
        for layer,res in zip(self.layers, self.res):
            h = layer(h, A) + res(h)
        logits = self.classifier(h)
        return (logits,h) if return_features else logits

def ppr_matrix(A_dense, alpha=0.15, iters=10):
    """Approx Personalized PageRank: P = a*(I - (1-a)Ahat)^-1, power-iterated."""
    N = A_dense.size(0); dev = A_dense.device
    deg = A_dense.sum(1, keepdim=True).clamp(min=1.0)
    Ahat = A_dense / deg                        # row-normalized
    P = torch.eye(N, device=dev)
    cur = torch.eye(N, device=dev)
    for _ in range(iters):
        cur = (1-alpha) * (Ahat @ cur)
        P = P + cur
    return alpha * P

def renode_weights(A_dense, y, ncls, w_min=0.5, w_max=1.5):
    """
    ReNode per-node weight from topology conflict.
    Totoro = sum over classes c of PPR-mass leaking into other classes.
    High conflict (near boundary) -> lower weight.
    """
    N = A_dense.size(0); dev = A_dense.device
    P = ppr_matrix(A_dense)                     # (N,N)
    onehot = F.one_hot(y, ncls).float()         # (N,C)
    class_mass = P @ onehot                      # (N,C) PPR mass per class
    self_mass = class_mass.gather(1, y.view(-1,1)).squeeze(1)
    total = class_mass.sum(1).clamp(min=1e-9)
    conflict = 1.0 - (self_mass/total)          # in [0,1], high = boundary
    # rank-based cosine schedule: lowest conflict -> w_max
    order = torch.argsort(conflict)             # ascending conflict
    ranks = torch.zeros(N, device=dev)
    ranks[order] = torch.arange(N, device=dev, dtype=torch.float)
    r = ranks/max(N-1,1)
    w = w_min + 0.5*(w_max-w_min)*(1+torch.cos(np.pi*r))   # rank0->w_max
    return w

def make_class_weight(train_dl, label_map):
    """Inverse-frequency class weights from train CSV (for ReNode + TAM)."""
    df = pd.read_csv(cfg.TRAIN_CSV)
    counts = df[cfg.LABEL_COL].map(label_map).value_counts().sort_index()
    cw = (counts.sum()/(len(counts)*counts)).values
    return torch.tensor(cw, dtype=torch.float, device=cfg.DEVICE)

def make_renode_criterion(train_dl, label_map):
    cw = make_class_weight(train_dl, label_map)
    return ("renode", cw)                        # carried into step_fn

def renode_step(model, imgs, labels, criterion):
    cw = criterion[1]
    logits, _ = model(imgs, return_features=True)
    A = model.last_adj.to_dense()
    rw = renode_weights(A, labels, cfg.NUM_CLASSES)       # (B,) topology weight
    ce = F.cross_entropy(logits, labels, weight=cw, reduction="none")
    loss = (rw * ce).mean()
    return loss, logits, labels

run_baseline("ReNode", lambda ncls: ReNodeGCN(ncls),
             make_criterion=make_renode_criterion, step_fn=renode_step)


########################################################
#  ReNode  |  RUN 1/1  |  SEED=42
########################################################
  Train:23234 Val:4085 Test:6873 | Classes:40
  Trainable params: 2,646,568

  Training 50 epochs


  Epoch   1: 100%|███████████████████████████████████| 364/364 [01:35<00:00,  3.80it/s, loss=6.8783, acc=38.7%]


  Ep   1/50  TrLoss=2.1814 TrAcc=38.7%  ValLoss=1.6281 ValF1=0.5045 ValAcc=52.4%  LR=9.99e-04 [109s]
  ★ best F1=0.5045


  Epoch   2: 100%|███████████████████████████████████| 364/364 [01:36<00:00,  3.76it/s, loss=3.1327, acc=54.7%]


  Ep   2/50  TrLoss=1.3863 TrAcc=54.7%  ValLoss=1.6182 ValF1=0.5296 ValAcc=53.4%  LR=9.96e-04 [110s]
  ★ best F1=0.5296


...

  Ep  49/50  TrLoss=0.3782 TrAcc=85.0%  ValLoss=0.5336 ValF1=0.8051 ValAcc=81.2%  LR=9.87e-07 [109s]


  Epoch  50: 100%|███████████████████████████████████| 364/364 [01:37<00:00,  3.75it/s, loss=2.4701, acc=85.0%]


  Ep  50/50  TrLoss=0.3725 TrAcc=85.0%  ValLoss=0.4951 ValF1=0.8114 ValAcc=82.1%  LR=0.00e+00 [110s]
  ★ best F1=0.8114

────────────────────────────────────────────────────
  VAL [Ep 50] METRICS
────────────────────────────────────────────────────
  Loss                        0.4951
  Accuracy                    82.06%
  Precision (Macro)           0.8175
  Recall (Macro)              0.8163
  F1 (Macro)                  0.8114
  F1 (Weighted)               0.8185
  ROC-AUC (OvR)               0.9949
  ROC-AUC (OvO)               0.9949
  MCC                         0.8159
────────────────────────────────────────────────────
  Loaded best (ep 50, F1=0.8114)



────────────────────────────────────────────────────
  TEST METRICS
────────────────────────────────────────────────────
  Loss                        0.4883
  Accuracy                    82.07%
  Precision (Macro)           0.8193
  Recall (Macro)              0.8208
  F1 (Macro)                  0.8149
  F1 (Weighted)               0.8182
  ROC-AUC (OvR)               0.9952
  ROC-AUC (OvO)               0.9952
  MCC                         0.8161
────────────────────────────────────────────────────

  CLASSIFICATION REPORT
                                                                                            precision    recall  f1-score   support
  
                                                                 Acute Cerebellitis in HIV       0.62      0.40      0.49       196
                                                      Acute Unilateral Cerebellitis in HIV       0.35      0.43      0.39        99
                                                              Adenom

[{'seed': 42,
  'accuracy': 0.8207478539211407,
  'f1_macro': 0.8148903842196837,
  'precision_macro': 0.8192545005119293,
  'recall_macro': 0.8208261349968516,
  'precision_weighted': 0.8257407810008615,
  'recall_weighted': 0.8207478539211407,
  'f1_weighted': 0.8181698872290807,
  'f1_per_class': array([0.48598131, 0.38565022, 0.96594427, 0.81681682, 0.44725738,
         0.94957983, 0.95238095, 0.86192469, 0.95412844, 0.9047619 ,
         0.49855072, 0.96725441, 0.76294278, 0.94974874, 0.95428571,
         0.77927928, 0.74479167, 0.93602694, 0.91596639, 0.48854962,
         0.56880734, 0.83544304, 0.92879257, 0.81690141, 0.85087719,
         0.83587786, 0.97457627, 0.89655172, 0.95798319, 0.87096774,
         0.71947195, 0.7606264 , 0.95424837, 1.        , 0.93059126,
         0.96819788, 0.64573991, 0.74193548, 0.7037037 , 0.9125    ]),
  'roc_auc_ovr': np.float64(0.995213572336057),
  'roc_auc_ovo': np.float64(0.995213572336057),
  'roc_auc': np.float64(0.995213572336057),
  'mcc'

In [5]:
# ══════════════════════════════════════════════════════════════
#  CELL 5: TAM   (Song et al., ICML 2022)
#  "TAM: Topology-Adaptive Margin Loss for Class-Imbalanced
#   Node Classification"
#  GCN encoder + topology-adaptive margin: per-node logits adjusted by
#  (a) class-wise connectivity (ACM, anomalous connectivity margin) and
#  (b) class-frequency margin (LDAM-style), driven by the batch graph.
# ══════════════════════════════════════════════════════════════
class TAMGCN(nn.Module):
    def __init__(self, ncls):
        super().__init__()
        d,H = cfg.FEATURE_DIM, cfg.GNN_HIDDEN_DIM
        self.backbone = FrozenBackbone(cfg.BACKBONE)
        dims=[d]+[H]*cfg.GNN_LAYERS
        self.layers = nn.ModuleList([GCNLayer(dims[i],dims[i+1]) for i in range(cfg.GNN_LAYERS)])
        self.res = nn.ModuleList([nn.Linear(dims[i],dims[i+1],bias=False) if dims[i]!=dims[i+1]
                                  else nn.Identity() for i in range(cfg.GNN_LAYERS)])
        self.classifier = nn.Linear(H, ncls)
        self.last_adj = None
    def forward(self, x, return_features=False):
        h = self.backbone(x)
        A = build_knn_graph(h.detach(), k=BASELINE_K, sym=True, norm=True)
        self.last_adj = A
        for layer,res in zip(self.layers, self.res):
            h = layer(h, A) + res(h)
        logits = self.classifier(h)
        return (logits,h) if return_features else logits

def make_tam_criterion(train_dl, label_map):
    df = pd.read_csv(cfg.TRAIN_CSV)
    counts = df[cfg.LABEL_COL].map(label_map).value_counts().sort_index().values.astype(float)
    cls_num = torch.tensor(counts, dtype=torch.float, device=cfg.DEVICE)
    # LDAM-style class margin: m_c ∝ 1 / n_c^{1/4}
    cmargin = 1.0 / torch.sqrt(torch.sqrt(cls_num))
    cmargin = cmargin / cmargin.max()           # normalize to [.,1]
    cw = (cls_num.sum()/(len(cls_num)*cls_num))  # inv-freq class weight
    return ("tam", cmargin, cw)

def tam_step(model, imgs, labels, criterion):
    _, cmargin, cw = criterion
    logits, _ = model(imgs, return_features=True)
    A = model.last_adj.to_dense()
    N = logits.size(0); ncls = cfg.NUM_CLASSES; dev = logits.device

    # ── ACM: topology-aware connectivity margin ──────────────
    # neighbor class distribution per node (row-normalized adj @ onehot)
    deg = A.sum(1, keepdim=True).clamp(min=1.0)
    Arw = A / deg
    onehot = F.one_hot(labels, ncls).float()
    nbr_dist = Arw @ onehot                      # (N,C) neighbor class mass
    # node's own-class connectivity vs other-class connectivity
    self_conn = nbr_dist.gather(1, labels.view(-1,1)).squeeze(1)   # (N,)
    # anomaly: low own-class connectivity -> needs larger margin
    acm = (1.0 - self_conn).clamp(0,1)           # (N,) in [0,1]

    # ── combine class margin + topology margin into logit shift ──
    base_m = cmargin[labels]                      # (N,) class freq margin
    tam_margin = base_m * (1.0 + acm)             # topology-adaptive scale
    shift = torch.zeros_like(logits)
    shift.scatter_(1, labels.view(-1,1), tam_margin.view(-1,1).to(logits.dtype))
    logits_adj = logits - shift                   # enlarge margin on true class

    loss = F.cross_entropy(logits_adj, labels, weight=cw.to(logits.dtype))
    return loss, logits, labels                   # acc on unshifted logits

run_baseline("TAM", lambda ncls: TAMGCN(ncls),
             make_criterion=make_tam_criterion, step_fn=tam_step)


########################################################
#  TAM  |  RUN 1/1  |  SEED=42
########################################################
  Train:23234 Val:4085 Test:6873 | Classes:40
  Trainable params: 2,646,568

  Training 50 epochs


  Epoch   1: 100%|███████████████████████████████████| 364/364 [02:04<00:00,  2.93it/s, loss=4.8778, acc=41.1%]


  Ep   1/50  TrLoss=3.4383 TrAcc=41.1%  ValLoss=1.5914 ValF1=0.5268 ValAcc=54.3%  LR=9.99e-04 [146s]
  ★ best F1=0.5268


  Epoch   2: 100%|███████████████████████████████████| 364/364 [01:32<00:00,  3.94it/s, loss=6.9534, acc=56.9%]


  Ep   2/50  TrLoss=2.3726 TrAcc=56.9%  ValLoss=1.3822 ValF1=0.5934 ValAcc=60.5%  LR=9.96e-04 [105s]
  ★ best F1=0.5934


...

  Ep  49/50  TrLoss=0.7708 TrAcc=85.5%  ValLoss=0.5674 ValF1=0.8107 ValAcc=81.7%  LR=9.87e-07 [108s]


  Epoch  50: 100%|███████████████████████████████████| 364/364 [01:30<00:00,  4.02it/s, loss=4.4116, acc=85.4%]


  Ep  50/50  TrLoss=0.7682 TrAcc=85.4%  ValLoss=0.5261 ValF1=0.8146 ValAcc=82.2%  LR=0.00e+00 [103s]
  ★ best F1=0.8146

────────────────────────────────────────────────────
  VAL [Ep 50] METRICS
────────────────────────────────────────────────────
  Loss                        0.5261
  Accuracy                    82.20%
  Precision (Macro)           0.8195
  Recall (Macro)              0.8209
  F1 (Macro)                  0.8146
  F1 (Weighted)               0.8209
  ROC-AUC (OvR)               0.9950
  ROC-AUC (OvO)               0.9950
  MCC                         0.8174
────────────────────────────────────────────────────
  Loaded best (ep 50, F1=0.8146)



────────────────────────────────────────────────────
  TEST METRICS
────────────────────────────────────────────────────
  Loss                        0.5249
  Accuracy                    82.60%
  Precision (Macro)           0.8232
  Recall (Macro)              0.8284
  F1 (Macro)                  0.8203
  F1 (Weighted)               0.8230
  ROC-AUC (OvR)               0.9953
  ROC-AUC (OvO)               0.9953
  MCC                         0.8215
────────────────────────────────────────────────────

  CLASSIFICATION REPORT
                                                                                            precision    recall  f1-score   support
  
                                                                 Acute Cerebellitis in HIV       0.57      0.27      0.36       196
                                                      Acute Unilateral Cerebellitis in HIV       0.32      0.51      0.39        99
                                                              Adenom

[{'seed': 42,
  'accuracy': 0.8259857413065619,
  'f1_macro': 0.8203350032598739,
  'precision_macro': 0.8232298490631382,
  'recall_macro': 0.8284192132110058,
  'precision_weighted': 0.8308832442163616,
  'recall_weighted': 0.8259857413065619,
  'f1_weighted': 0.823047001553711,
  'f1_per_class': array([0.36111111, 0.38910506, 0.96594427, 0.84161491, 0.4940239 ,
         0.95435685, 0.9556314 , 0.88429752, 0.95852535, 0.89328063,
         0.52247191, 0.96984925, 0.77659574, 0.94974874, 0.95726496,
         0.79176201, 0.75064267, 0.94915254, 0.92827004, 0.4973822 ,
         0.55948553, 0.84183673, 0.9382716 , 0.82967033, 0.85087719,
         0.84291188, 0.9707113 , 0.89878543, 0.96638655, 0.87037037,
         0.72483221, 0.7706422 , 0.96732026, 1.        , 0.92346939,
         0.97902098, 0.69230769, 0.75471698, 0.72592593, 0.9148265 ]),
  'roc_auc_ovr': np.float64(0.9952830873603071),
  'roc_auc_ovo': np.float64(0.9952830873603071),
  'roc_auc': np.float64(0.9952830873603071),
  'mc